In [31]:
import numpy as np

# Raw features (15 samples, 3 features each)
hours_studied = np.array([1.2, 1.5, 2.0, 2.2, 2.8, 3.0, 3.2, 3.5, 4.0, 4.2, 4.5, 4.8, 5.0, 5.2, 5.5], dtype=np.float64)
hours_slept   = np.array([5.0, 6.0, 5.5, 6.5, 7.0, 6.0, 7.5, 7.0, 8.0, 7.0, 8.0, 7.5, 8.0, 6.5, 8.0], dtype=np.float64)
attendance    = np.array([0.6, 0.7, 0.5, 0.8, 0.6, 0.9, 0.7, 0.85, 0.9, 0.95, 0.8, 0.9, 1.0, 0.75, 0.95], dtype=np.float64)

# Labels: 0 = failed, 1 = passed
passed = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=np.float64)

# Build design matrix X (bias + features)
X = np.column_stack([np.ones(len(hours_studied)), hours_studied, hours_slept, attendance])

# Reshape labels to column vector
y = passed.reshape(-1, 1)

# Sigmoid function
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# Forward pass
def forward(X, w):
    z = X @ w
    p_hat = sigmoid(z)
    return p_hat

# Binary Cross Entropy Loss
def binary_cross_entropy(y, p_hat, eps=1e-15):
    p_hat = np.clip(p_hat, eps, 1 - eps)
    loss = -np.mean(y*np.log(p_hat) + (1-y)*np.log(1-p_hat))
    return loss

# Gradient of loss
def gradient(X, y, p_hat):
    n = X.shape[0]
    grad = (1/n) * X.T @ (p_hat - y)
    return grad
#---------------------------------------------------------------------------
w = np.zeros((4,1))

# Forward pass (prediction probabilities)
p_hat = forward(X, w)

# Calculate loss
loss = binary_cross_entropy(y, p_hat)

# Calculate gradient
grad = gradient(X, y, p_hat)

# Print results
print("Predicted probabilities:\n", p_hat)
print("\nLoss:", loss)
print("\nGradient:\n", grad)

Predicted probabilities:
 [[0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]]

Loss: 0.6931471805599453

Gradient:
 [[-0.16666667]
 [-1.10666667]
 [-1.45      ]
 [-0.18333333]]


In [27]:
# Training loop
np.random.seed(42)

learning_rate = 0.1
epochs = 1000

n_features = X.shape[1]
w = np.random.randn(n_features, 1) * 0.1

loss_history = []

for epoch in range(epochs):

    # Forward pass
    p_hat = forward(X, w)

    # Compute loss
    loss = binary_cross_entropy(y, p_hat)
    loss_history.append(loss)

    # Compute gradient
    grad = gradient(X, y, p_hat)

    # Update weights
    w = w - learning_rate * grad

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

Epoch 200, Loss: 0.1884
Epoch 400, Loss: 0.1404
Epoch 600, Loss: 0.1189
Epoch 800, Loss: 0.1060
Epoch 1000, Loss: 0.0971


In [28]:
p_hat = forward(X, w)

y_pred = (p_hat >= 0.5).astype(np.float64)

accuracy = np.mean(y_pred == y)

print("Training accuracy:", accuracy)


Training accuracy: 1.0


In [29]:
new_student = np.array([[1, 3.8, 7, 0.88]])
p = forward(new_student, w)
print("P(pass) =", p[0,0], "->", "Pass" if p[0,0] >= 0.5 else "Fail")

P(pass) = 0.98522498199886 -> Pass


In [30]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# Same 15-student data as the NumPy exercise
hours_studied = np.array([1.2, 1.5, 2.0, 2.2, 2.8, 3.0, 3.2, 3.5, 4.0, 4.2, 4.5, 4.8, 5.0, 5.2, 5.5], dtype=np.float64)
hours_slept   = np.array([5.0, 6.0, 5.5, 6.5, 7.0, 6.0, 7.5, 7.0, 8.0, 7.0, 8.0, 7.5, 8.0, 6.5, 8.0], dtype=np.float64)
attendance    = np.array([0.6, 0.7, 0.5, 0.8, 0.6, 0.9, 0.7, 0.85, 0.9, 0.95, 0.8, 0.9, 1.0, 0.75, 0.95], dtype=np.float64)
passed        = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=np.int64)

# Build X matrix
X = np.column_stack((hours_studied, hours_slept, attendance))

# Labels
y = passed

print("X shape:", X.shape, "  y shape:", y.shape)

# Create and train model
model = LogisticRegression(fit_intercept=True, random_state=42)
model.fit(X, y)

# Predictions
y_pred = model.predict(X)

# Probabilities of passing
y_proba = model.predict_proba(X)[:, 1]

# Accuracy
accuracy = (y_pred == y).mean()
print("Training accuracy:", accuracy)

# Model parameters
print("Intercept:", model.intercept_[0])
print("Coefficients (hours_studied, hours_slept, attendance):", model.coef_[0])

# Predict new student
new_student = np.array([[3.8, 7, 0.88]])

pred = model.predict(new_student)
proba = model.predict_proba(new_student)[0, 1]

print(f"New student → P(pass) = {proba:.3f} → {'Pass' if pred[0] == 1 else 'Fail'}")

X shape: (15, 3)   y shape: (15,)
Training accuracy: 0.9333333333333333
Intercept: -7.0193345650524455
Coefficients (hours_studied, hours_slept, attendance): [1.40681377 0.45640094 0.24651745]
New student → P(pass) = 0.850 → Pass
